# Introdução à Geometria de Posição

**Módulo 06 — Geometria Espacial** · Notebook 01 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0601a_introducao_geometria_de_posicao.ipynb)

**Pré-requisitos**
- Módulo 00 — Fundamentos e Python (todo): [`00_introducao_python.ipynb`](../00_fundamentos_e_python/00_introducao_python.ipynb)
- Módulo 04 — Geometria Plana: [`0401a_nocoes_e_proposicoes_primitivas.ipynb`](../04_geometria_plana/0401a_nocoes_e_proposicoes_primitivas.ipynb) (ponto, reta, plano e postulados), [`0403a_angulos.ipynb`](../04_geometria_plana/0403a_angulos.ipynb) (ângulos) e [`0406a_paralelismo.ipynb`](../04_geometria_plana/0406a_paralelismo.ipynb) (posições relativas de retas no plano)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- reconhecer ponto, reta e plano como conceitos primitivos e usar a notação correta para cada um;
- usar os postulados para decidir quando pontos e retas determinam um único plano;
- classificar a posição relativa de duas retas no espaço, inclusive retas reversas;
- descrever o que acontece quando dois planos distintos se encontram.

**Tempo estimado:** 60 a 80 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o arquivo `utils_3d.py`, que guarda todas as funções de desenho deste módulo.

Você não precisa saber como esse arquivo funciona por dentro para estudar a matemática. Mas, se você gosta de programar, abra o `utils_3d.py` e dê uma olhada: o código é comentado e você pode modificá-lo à vontade.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets

## Por que a mesa de três pernas nunca balança?

Você já sentou num restaurante e a mesa ficou mancando, indo para um lado e para o outro? Quase sempre ela tem **quatro** pernas. Agora repare num tripé de câmera, num banquinho de ordenhar vaca ou numa mesa de três pernas: eles **nunca** balançam, nem em chão irregular.

Isso não é sorte nem engenharia caprichada. É matemática: **três pontos (que não estejam alinhados) determinam um único plano**. As pontas das três pernas sempre conseguem se apoiar juntas no chão. Com quatro pernas, a quarta ponta precisa "concordar" com o plano das outras três, e qualquer milímetro de diferença faz a mesa mancar.

🕹️ Mexa no controle abaixo: ele muda o comprimento da quarta perna de uma mesa. O plano azul é o plano definido pelas pontas das outras três pernas.

In [ ]:
# As coordenadas (x, y, z) abaixo servem só para posicionar o desenho.
ALTURA_DA_MESA = 2
LADO_DO_TAMPO = 3

# Os quatro cantos do tampo, vistos de cima, na ordem das pernas 1, 2, 3 e 4.
cantos_do_tampo = [(-1.5, -1.5), (1.5, -1.5), (1.5, 1.5), (-1.5, 1.5)]


def desenhar_mesa(diferenca: float) -> None:
    """Desenha a mesa com a ponta da perna 4 na altura `diferenca`."""
    fig = u3d.criar_figura(titulo="Uma mesa de quatro pernas")

    # Tampo: um pedaço de plano horizontal, com as quatro bordas.
    fig = u3d.desenhar_plano(
        fig, ponto=(0, 0, ALTURA_DA_MESA), normal=(0, 0, 1), tamanho=LADO_DO_TAMPO,
        cor=u3d.CORES["solido"], opacidade=0.5, mostrar_borda=False,
    )
    for i in range(4):
        x1, y1 = cantos_do_tampo[i]
        x2, y2 = cantos_do_tampo[(i + 1) % 4]  # % 4 faz o canto 4 se ligar ao 1
        fig = u3d.desenhar_segmento(
            fig, (x1, y1, ALTURA_DA_MESA), (x2, y2, ALTURA_DA_MESA),
            cor=u3d.CORES["solido"], espessura=5,
        )

    # Pernas: as três primeiras chegam ao chão (altura 0); a quarta para na
    # altura `diferenca` (acima do chão se for mais curta, abaixo se for mais longa).
    pontas = []
    for numero, (x, y) in enumerate(cantos_do_tampo, start=1):
        altura_da_ponta = diferenca if numero == 4 else 0
        fig = u3d.desenhar_segmento(
            fig, (x, y, ALTURA_DA_MESA), (x, y, altura_da_ponta),
            cor=u3d.CORES["apoio"], espessura=8,
        )
        pontas.append((x, y, altura_da_ponta))

    # O plano determinado pelas pontas das pernas 1, 2 e 3.
    fig = u3d.desenhar_plano_por_tres_pontos(
        fig, pontas[0], pontas[1], pontas[2], cor=u3d.CORES["plano"], opacidade=0.3
    )

    # As pontas das pernas; a da perna 4 em destaque.
    fig = u3d.desenhar_pontos(fig, pontas[:3], rotulos=["1", "2", "3"])
    fig = u3d.desenhar_ponto(
        fig, pontas[3], rotulo="4", cor=u3d.CORES["destaque"], tamanho=8
    )
    u3d.mostrar(fig)

    # Comparamos com uma tolerância porque o slider pode gerar 0.0000000001 no
    # lugar de 0.
    if abs(diferenca) < 1e-9:
        print("As quatro pontas estão no mesmo plano: a mesa fica firme.")
    else:
        print("A ponta da perna 4 está fora do plano das outras três: a mesa manca!")


interact(
    desenhar_mesa,
    diferenca=widgets.FloatSlider(
        min=-0.5, max=0.5, step=0.05, value=0.3, description="Perna 4",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=0.3, continuous_update=False, description='Perna 4', max=0.5, min=-0.5…

Neste notebook você vai entender **por que** isso acontece, e vai ver que no espaço as coisas são um pouco diferentes do que no plano. Por exemplo: no plano, duas retas que nunca se encontram são sempre paralelas. No espaço, não! Pense num viaduto passando por cima de uma avenida: as duas pistas nunca se cruzam, e mesmo assim não são paralelas. Vamos descobrir o nome disso.

## 1. Do plano para o espaço

Na Geometria Plana (módulo 04), tudo acontecia numa folha de papel: triângulos, circunferências, retas... Todas as figuras cabiam num único plano.

Agora vamos sair da folha. A **Geometria Espacial** estuda figuras que ocupam o espaço: caixas, bolas, pirâmides, latas, e também as posições de pontos, retas e planos uns em relação aos outros.

> 📌 **Definição.** Uma figura é **plana** quando todos os seus pontos pertencem a um mesmo plano. Caso contrário, ela é uma figura **espacial** (ou **não plana**).

A parte da Geometria Espacial que estuda apenas as **posições** (sem medir comprimentos, áreas ou volumes) se chama **Geometria de Posição**. É por ela que começamos.

Veja abaixo um triângulo (figura plana) e uma "pirâmide de base triangular" (figura espacial). Use o mouse para **girar**, **aproximar** e **afastar** a figura.

> 💡 **Dica.** Todas as figuras 3D deste módulo podem ser giradas com o mouse (clique e arraste) e ampliadas com a rolagem. Gire sempre! Muitas coisas que parecem estranhas num desenho parado ficam óbvias quando você muda o ponto de vista.

In [3]:
# Os números dentro das tuplas são só posições para o desenho, no formato (x, y, z).
# Eles não são o assunto deste módulo: servem apenas para dizer ao computador onde
# cada ponto fica.

# Figura plana, à esquerda: o triângulo ABC.
A, B, C = (-5, -1, 0), (-2, -1, 0.5), (-3.5, 1.5, 1)

# Figura espacial, à direita: o tetraedro (pirâmide de base triangular) PQRS.
P, Q, R, S = (2, -1, 0), (5, -1, 0), (3.5, 1.5, 0), (3.5, 0, 2.5)

fig = u3d.criar_figura(titulo="Figura plana × figura espacial")

# O triângulo e o plano α que o contém.
fig = u3d.desenhar_plano_por_tres_pontos(fig, A, B, C, rotulo="α", opacidade=0.2)
for inicio, fim in [(A, B), (B, C), (C, A)]:
    fig = u3d.desenhar_segmento(fig, inicio, fim, cor=u3d.CORES["reta"], espessura=6)
fig = u3d.desenhar_pontos(fig, [A, B, C], rotulos=["A", "B", "C"])

# O tetraedro: seis arestas, sem plano nenhum (não existe um plano com os 4 vértices).
for inicio, fim in [(P, Q), (Q, R), (R, P), (P, S), (Q, S), (R, S)]:
    fig = u3d.desenhar_segmento(
        fig, inicio, fim, cor=u3d.CORES["reta_secundaria"], espessura=6
    )
fig = u3d.desenhar_pontos(fig, [P, Q, R, S], rotulos=["P", "Q", "R", "S"])

fig = u3d.adicionar_rotulo(fig, (-3.5, 0, 3), "figura plana", tamanho=16)
fig = u3d.adicionar_rotulo(fig, (3.5, 0, 3.5), "figura espacial", tamanho=16)
u3d.mostrar(fig)

**Pare e pense**

1. Uma folha de papel dobrada ao meio, meio aberta, é uma figura plana ou espacial?
2. Gire a pirâmide acima até que ela pareça um triângulo. Isso significa que ela é plana? Por quê?

<details>
<summary>Ver respostas</summary>

1. Espacial: as duas metades da folha ficam em planos diferentes.
2. Não. Ela só **parece** plana de um certo ponto de vista; o quarto vértice continua fora do plano dos outros três. Desenhos enganam, e é por isso que precisamos de definições precisas.
</details>

## 2. Conceitos primitivos: ponto, reta e plano

Tente definir **ponto** sem usar nenhuma outra palavra geométrica. Difícil, não? Se você disser "é uma posição no espaço", vai precisar definir "posição"; se definir posição, vai precisar de outra palavra, e assim por diante. É como procurar uma palavra no dicionário e cair num círculo infinito.

Para escapar desse círculo, a Geometria escolhe alguns conceitos que **não são definidos**: eles são aceitos a partir da nossa intuição. São os **conceitos primitivos**:

> 📌 **Conceitos primitivos.** **Ponto**, **reta** e **plano** não têm definição. Temos apenas uma ideia intuitiva deles:
> - **ponto**: não tem tamanho; pense na marca da ponta de um lápis muito fino;
> - **reta**: é infinita nos dois sentidos, não tem espessura e é "reta" (não faz curva); pense num fio esticado que nunca acaba;
> - **plano**: é infinito em todas as direções, não tem espessura e é "liso"; pense na superfície de uma mesa que se estende para sempre.
>
> O **espaço** é o conjunto de todos os pontos.

**Notação**

| Elemento | Como escrevemos | Exemplos |
|---|---|---|
| Ponto | letra latina maiúscula | $A$, $B$, $P$ |
| Reta | letra latina minúscula | $r$, $s$, $t$ |
| Plano | letra grega minúscula | $\alpha$ (alfa), $\beta$ (beta), $\gamma$ (gama), $\pi$ (pi) |

Para dizer que um ponto está numa reta ou num plano, usamos **pertence**: $A \in r$, $A \in \alpha$. Para dizer que uma reta está num plano, usamos **está contida**: $r \subset \alpha$.

> ⚠️ **Erro comum.** Escrever $r \in \alpha$. A reta não é um *elemento* do plano; ela é um *conjunto de pontos* que fica dentro dele. Ponto **pertence** ($\in$); reta **está contida** ($\subset$).

> 💡 **Dica.** Como não dá para desenhar algo infinito, representamos o plano por um **paralelogramo** (um "pedaço" dele) e a reta por um **trecho**. Nos nossos desenhos, os planos aparecem como quadrados semitransparentes. Lembre sempre: eles continuam além da borda.

In [4]:
# 1. Criamos uma figura vazia
fig = u3d.criar_figura(titulo="Ponto, reta e plano")

# 2. Desenhamos o plano α (um pedaço dele)
fig = u3d.desenhar_plano(fig, ponto=(0, 0, 0), normal=(0, 0, 1), rotulo="α")

# 3. Desenhamos a reta r, contida em α
fig = u3d.desenhar_reta(fig, ponto=(0, 0, 0), direcao=(1, 1, 0), rotulo="r")

# 4. Um ponto A que pertence a r, e um ponto B fora do plano
fig = u3d.desenhar_ponto(fig, (1, 1, 0), rotulo="A")
fig = u3d.desenhar_ponto(fig, (-1, 1.5, 2), rotulo="B", cor=u3d.CORES["destaque"])

# 5. Mostramos o resultado
u3d.mostrar(fig)

Repare no padrão: toda função `u3d.desenhar_...` recebe a figura, acrescenta algo a ela e devolve a figura atualizada. Por isso escrevemos `fig = u3d.desenhar_...(fig, ...)`.

Quer saber o que uma função faz e quais parâmetros ela aceita? Use `help`:

In [5]:
help(u3d.desenhar_plano)

Help on function desenhar_plano in module utils_3d:

desenhar_plano(fig: 'go.Figure', ponto: 'Vetor3D', normal: 'Vetor3D', rotulo: 'str | None' = None, cor: 'str | None' = None, opacidade: 'float' = 0.35, tamanho: 'float' = 6, mostrar_borda: 'bool' = True) -> 'go.Figure'
    Desenha (um pedaço de) um plano.
    
    Um plano é infinito em todas as direções. Como não dá para desenhar isso, a
    função desenha um "pedaço" quadrado do plano, de lado `tamanho`, centrado em
    `ponto`. É a mesma ideia dos livros, que representam um plano por um
    paralelogramo. Lembre-se: o plano continua além da borda.
    
    A posição do plano é dada por um vetor `normal`, que é perpendicular a ele (como
    um mastro fincado no chão é perpendicular ao chão).
    
    Parâmetros:
        fig: a figura.
        ponto: um ponto do plano, como (x, y, z). O quadrado fica centrado nele.
        normal: vetor perpendicular ao plano. (0, 0, 1) dá um plano horizontal.
        rotulo: nome do plano, normalme

**Pare e pense**

Com base na figura acima, escreva com símbolos:
1. a relação entre $A$ e $r$;
2. a relação entre $r$ e $\alpha$;
3. a relação entre $B$ e $\alpha$.

<details>
<summary>Ver respostas</summary>

1. $A \in r$
2. $r \subset \alpha$
3. $B \notin \alpha$
</details>

## 3. Postulados: as regras do jogo

Se ponto, reta e plano não têm definição, como provamos alguma coisa sobre eles? Usando **postulados** (ou **axiomas**): afirmações aceitas como verdadeiras sem demonstração. Eles são as "regras do jogo" da Geometria. Tudo o que vier depois (os **teoremas**) será demonstrado a partir delas.

Antes, duas definições:

> 📌 **Definição.** Pontos são **colineares** quando pertencem a uma mesma reta. Pontos (ou retas) são **coplanares** quando pertencem a (ou estão contidos em) um mesmo plano.

**Postulado da existência**
> Numa reta, bem como fora dela, há infinitos pontos. Num plano, bem como fora dele, há infinitos pontos.

**Postulado da determinação da reta**
> Dois pontos distintos determinam uma única reta, que passa por eles.

**Postulado da determinação do plano**
> Três pontos não colineares determinam um único plano, que passa por eles.

**Postulado da inclusão**
> Se uma reta tem dois pontos distintos num plano, então ela está contida nesse plano.

"Determinar" quer dizer duas coisas ao mesmo tempo: **existe** e é **único**. Existe uma reta passando por $A$ e $B$, e não existe outra.

> 🕵️ **Curiosidade histórica.** Por volta de 300 a.C., o grego **Euclides** escreveu *Os Elementos*, que organizou a Geometria a partir de definições e postulados; os livros XI a XIII tratam justamente do espaço. Mais de dois mil anos depois, em 1899, o alemão **David Hilbert** reescreveu esses fundamentos com todo rigor. Conta-se que ele dizia que, em vez de "pontos, retas e planos", poderíamos falar em "mesas, cadeiras e canecas", desde que obedecessem aos postulados. O que importa não é o que as coisas *são*, mas as regras que elas seguem.

### O postulado da inclusão em ação

🕹️ Abaixo, os pontos $A$ e $B$ começam dentro do plano $\alpha$ e, portanto, a reta $AB$ está inteira dentro dele. Use o controle para levantar o ponto $B$ e veja o que acontece com a reta.

In [6]:
def desenhar_inclusao(altura_de_B: float) -> None:
    """Desenha a reta AB com A fixo no plano α e B na altura escolhida."""
    ponto_a = (-1.5, -1, 0)
    ponto_b = (1.5, 1, altura_de_B)
    b_esta_no_plano = altura_de_B == 0

    fig = u3d.criar_figura(titulo="Postulado da inclusão")
    fig = u3d.desenhar_plano(fig, ponto=(0, 0, 0), normal=(0, 0, 1), rotulo="α")

    # A reta fica em destaque quando "fura" o plano.
    cor_da_reta = u3d.CORES["reta"] if b_esta_no_plano else u3d.CORES["destaque"]
    fig = u3d.desenhar_reta_por_dois_pontos(
        fig, ponto_a, ponto_b, rotulo="AB", cor=cor_da_reta, comprimento=8, espessura=5
    )
    fig = u3d.desenhar_pontos(fig, [ponto_a, ponto_b], rotulos=["A", "B"])
    u3d.mostrar(fig)

    if b_esta_no_plano:
        print(
            "A e B estão em α, então a reta AB está contida em α "
            "(postulado da inclusão)."
        )
    else:
        print(
            "B saiu de α: agora a reta AB tem só o ponto A em comum com α. "
            "Dizemos que ela 'fura' o plano."
        )


interact(
    desenhar_inclusao,
    altura_de_B=widgets.FloatSlider(
        min=0, max=3, step=0.25, value=0, description="altura de B",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='altura de B', max=3.0, step…

### Quantas retas e quantos planos?

Imagine alguns pontos espalhados no espaço, **sem três deles alinhados** e **sem quatro deles no mesmo plano**. Pelos postulados:
- cada **par** de pontos determina uma reta;
- cada **trio** de pontos determina um plano.

Quantas retas e quantos planos aparecem? Em vez de contar à mão, vamos deixar o computador listar todos os pares e trios.

In [7]:
def contar_retas_e_planos(n: int) -> tuple[int, int]:
    """Conta as retas e os planos determinados por n pontos do espaço.

    Supõe que não há três pontos alinhados nem quatro pontos no mesmo plano.
    Cada par de pontos (i < j) determina uma reta; cada trio (i < j < k), um plano.
    """
    retas = 0
    for i in range(n):
        for j in range(i + 1, n):  # j começa depois de i: cada par conta uma vez só
            retas += 1

    planos = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):  # i < j < k: cada trio conta uma vez só
                planos += 1

    return retas, planos


# Tabela para 2 a 8 pontos
print(f"{'pontos':>6} | {'retas':>5} | {'planos':>6}")
print("-" * 24)
for n in range(2, 9):
    retas, planos = contar_retas_e_planos(n)
    print(f"{n:>6} | {retas:>5} | {planos:>6}")

pontos | retas | planos
------------------------
     2 |     1 |      0
     3 |     3 |      1
     4 |     6 |      4
     5 |    10 |     10
     6 |    15 |     20
     7 |    21 |     35
     8 |    28 |     56


Veja o caso de **4 pontos**: eles formam um tetraedro. Cada par de vértices dá uma reta (as arestas) e cada trio dá um plano (as faces).

In [8]:
# Quatro pontos sem três alinhados e sem os quatro no mesmo plano.
vertices_tetraedro = [(0, 0, 0), (3, 0, 0), (1.5, 2.6, 0), (1.5, 0.9, 2.4)]
nomes = ["A", "B", "C", "D"]
cores_dos_planos = [
    u3d.CORES["plano"],
    u3d.CORES["plano_secundario"],
    u3d.CORES["plano_terciario"],
    u3d.CORES["destaque"],
]

fig = u3d.criar_figura(titulo="4 pontos: 6 retas e 4 planos")

# Os 4 planos: um para cada trio de pontos (laços aninhados, como na tabela).
numero_do_plano = 0
for i in range(4):
    for j in range(i + 1, 4):
        for k in range(j + 1, 4):
            fig = u3d.desenhar_plano_por_tres_pontos(
                fig, vertices_tetraedro[i], vertices_tetraedro[j],
                vertices_tetraedro[k], cor=cores_dos_planos[numero_do_plano],
                opacidade=0.15, tamanho=1, mostrar_borda=False,
            )
            numero_do_plano += 1

# As 6 retas: uma para cada par de pontos (desenhadas como segmentos).
for i in range(4):
    for j in range(i + 1, 4):
        fig = u3d.desenhar_segmento(
            fig, vertices_tetraedro[i], vertices_tetraedro[j],
            cor=u3d.CORES["reta"], espessura=6,
        )

fig = u3d.desenhar_pontos(fig, vertices_tetraedro, rotulos=nomes)
u3d.mostrar(fig)

**Pare e pense**

1. Olhando a tabela: com 4 pontos, quantas retas e quantos planos? Confira na figura.
2. Por que a contagem só funciona se nenhum trio de pontos for colinear?

<details>
<summary>Ver respostas</summary>

1. 6 retas e 4 planos (as 6 arestas e as 4 faces do tetraedro).
2. Se três pontos fossem colineares, os três pares formariam a **mesma** reta (contaríamos 3 retas onde existe 1), e esse trio não determinaria plano nenhum.
</details>

> 📎 **Conexão.** Existe uma fórmula para essas contagens sem precisar listar tudo. Você vai aprendê-la no módulo 08 (Análise Combinatória e Probabilidade).

## 4. Determinação de um plano

O postulado diz que três pontos não colineares determinam um plano. A partir dele, conseguimos provar que existem **quatro maneiras** de determinar um plano:

| Modo | O que é dado | Por que funciona |
|---|---|---|
| 1 | Três pontos não colineares | É o próprio postulado. |
| 2 | Uma reta e um ponto fora dela | Escolha dois pontos da reta; com o ponto de fora, temos três pontos não colineares. |
| 3 | Duas retas concorrentes | Pegue o ponto de encontro e mais um ponto de cada reta: três pontos não colineares. |
| 4 | Duas retas paralelas distintas | Por definição, retas paralelas estão num mesmo plano; e só existe um, porque ele precisa conter uma reta e um ponto da outra (modo 2). |

🎯 **Aplicações no dia a dia**
- **Tripé de câmera** e **mesa de três pernas**: modo 1.
- **Porta**: as dobradiças ficam numa reta (o eixo da porta). Sozinha, a porta gira à vontade em volta desse eixo. Quando você tranca, o trinco é um ponto fora do eixo, e a porta fica presa num único plano: modo 2.

🕹️ Escolha o modo no menu abaixo para ver cada caso.

In [ ]:
def no_plano_inclinado(a: float, b: float) -> tuple[float, float, float]:
    """Devolve um ponto de um plano inclinado fixo, usado só para o desenho.

    Andamos `a` unidades numa direção horizontal e `b` unidades numa direção que
    sobe enquanto avança. Todos os pontos gerados ficam no mesmo plano inclinado.
    """
    return (a, b, 0.7 * b)


MODOS = [
    "1 · Três pontos não colineares",
    "2 · Reta e ponto fora dela",
    "3 · Duas retas concorrentes",
    "4 · Duas retas paralelas distintas",
]


def desenhar_modo(modo: str) -> None:
    """Desenha os elementos dados em cada modo e o plano α que eles determinam."""
    fig = u3d.criar_figura(titulo=modo)
    cor_r = u3d.CORES["reta"]
    cor_s = u3d.CORES["reta_secundaria"]
    cor_ponto = u3d.CORES["destaque"]

    if modo == MODOS[0]:
        A = no_plano_inclinado(-1.5, -1)
        B = no_plano_inclinado(1.5, -1)
        C = no_plano_inclinado(0, 1.5)
        fig = u3d.desenhar_plano_por_tres_pontos(
            fig, A, B, C, rotulo="α", opacidade=0.2
        )
        fig = u3d.desenhar_pontos(
            fig, [A, B, C], rotulos=["A", "B", "C"], cor=cor_ponto, tamanho=8
        )

    elif modo == MODOS[1]:
        A = no_plano_inclinado(-1, -1)
        B = no_plano_inclinado(1, -1)
        P = no_plano_inclinado(0, 1.5)
        fig = u3d.desenhar_plano_por_tres_pontos(
            fig, A, B, P, rotulo="α", opacidade=0.2
        )
        fig = u3d.desenhar_reta_por_dois_pontos(
            fig, A, B, rotulo="r", cor=cor_r, comprimento=6, espessura=6
        )
        fig = u3d.desenhar_ponto(fig, P, rotulo="P", cor=cor_ponto, tamanho=8)

    elif modo == MODOS[2]:
        P = no_plano_inclinado(0, 0)  # o ponto de encontro das duas retas
        ponto_de_r = no_plano_inclinado(2, -1)
        ponto_de_s = no_plano_inclinado(1, 1.5)
        fig = u3d.desenhar_plano_por_tres_pontos(
            fig, P, ponto_de_r, ponto_de_s, rotulo="α", opacidade=0.2
        )
        fig = u3d.desenhar_reta_por_dois_pontos(
            fig, P, ponto_de_r, rotulo="r", cor=cor_r, comprimento=6, espessura=6
        )
        fig = u3d.desenhar_reta_por_dois_pontos(
            fig, P, ponto_de_s, rotulo="s", cor=cor_s, comprimento=6, espessura=6
        )
        fig = u3d.desenhar_ponto(fig, P, rotulo="P", cor=cor_ponto, tamanho=8)

    else:
        A = no_plano_inclinado(-1, -1)
        B = no_plano_inclinado(1, -1)
        C = no_plano_inclinado(-1, 1)
        D = no_plano_inclinado(1, 1)
        fig = u3d.desenhar_plano_por_tres_pontos(
            fig, A, B, C, rotulo="α", opacidade=0.2
        )
        fig = u3d.desenhar_reta_por_dois_pontos(
            fig, A, B, rotulo="r", cor=cor_r, comprimento=6, espessura=6
        )
        fig = u3d.desenhar_reta_por_dois_pontos(
            fig, C, D, rotulo="s", cor=cor_s, comprimento=6, espessura=6
        )

    u3d.mostrar(fig)


interact(
    desenhar_modo,
    modo=widgets.Dropdown(
        options=MODOS, value=MODOS[0], description="Modo",
        layout=widgets.Layout(width="380px"),
    ),
);

interactive(children=(Dropdown(description='Modo', layout=Layout(width='380px'), options=('1 · Três pontos não…

> ⚠️ **Erro comum.** Achar que **quaisquer** três pontos determinam um plano. Se os três pontos forem **colineares**, há **infinitos** planos passando por eles, todos girando em torno da mesma reta, como as páginas de um livro aberto em torno da lombada. O mesmo vale para uma reta sozinha.

Esse conjunto de todos os planos que passam por uma reta se chama **feixe de planos**. 

🕹️ Gire o plano abaixo em torno da reta $r$ e repare: todos os pontos da reta continuam dentro dele, em qualquer posição.

In [ ]:
import math


def desenhar_feixe(angulo: int) -> None:
    """Desenha um dos planos do feixe de planos que contêm a reta r."""
    # Três pontos colineares sobre a reta r.
    A, B, C = (-2, 0, 0), (0, 0, 0), (2, 0, 0)

    # Seno e cosseno (módulo 05) aparecem aqui só para girar o desenho: o ponto
    # fora_de_r dá uma volta em torno de r conforme o ângulo muda.
    theta = math.radians(angulo)
    fora_de_r = (0, math.cos(theta), math.sin(theta))

    fig = u3d.criar_figura(
        titulo=f"Um dos infinitos planos que contêm r (ângulo: {angulo}°)"
    )

    # Referência: o plano na posição 0°, bem apagado.
    fig = u3d.desenhar_plano_por_tres_pontos(
        fig, A, C, (0, 1, 0), cor=u3d.CORES["apoio"], opacidade=0.1,
        mostrar_borda=False,
    )
    # O plano na posição escolhida: ele contém A, C e o ponto fora_de_r.
    fig = u3d.desenhar_plano_por_tres_pontos(fig, A, C, fora_de_r, rotulo="α")

    fig = u3d.desenhar_reta(
        fig, ponto=(0, 0, 0), direcao=(1, 0, 0), rotulo="r", comprimento=8, espessura=6
    )
    fig = u3d.desenhar_pontos(
        fig, [A, B, C], rotulos=["A", "B", "C"], cor=u3d.CORES["destaque"], tamanho=7
    )
    u3d.mostrar(fig)


interact(
    desenhar_feixe,
    angulo=widgets.IntSlider(
        min=0, max=180, step=15, value=30, description="ângulo (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=30, continuous_update=False, description='ângulo (°)', max=180, step=15)…

O próprio `utils_3d` "sabe" disso. Veja o que acontece quando pedimos um plano passando por três pontos colineares:

In [11]:
fig = u3d.criar_figura()

try:
    fig = u3d.desenhar_plano_por_tres_pontos(fig, (0, 0, 0), (1, 1, 1), (2, 2, 2))
except ValueError as erro:
    print(f"O computador recusou: {erro}")

O computador recusou: Os três pontos são colineares: eles não determinam um único plano.


**Pare e pense**

1. Uma reta e um ponto **que pertence** a ela determinam um plano?
2. Duas retas que nunca se encontram sempre determinam um plano? (Guarde essa pergunta: ela será respondida na próxima seção.)

<details>
<summary>Ver respostas</summary>

1. Não. O ponto não acrescenta nada: continuam passando infinitos planos pela reta.
2. Nem sempre. Se forem paralelas, sim (modo 4). Mas existem retas que nunca se encontram e **não** estão num mesmo plano: são as retas reversas.
</details>

## 5. Posições relativas de duas retas

No plano, duas retas distintas ou se cruzam ou são paralelas. No espaço aparece uma terceira possibilidade. Para classificar duas retas $r$ e $s$, faça duas perguntas:

**Pergunta 1: existe um plano que contém as duas?**

- **Sim (retas coplanares):**
  - **coincidentes**: são a mesma reta ($r = s$);
  - **concorrentes**: têm um único ponto em comum ($r \cap s = \{P\}$);
  - **paralelas**: estão num mesmo plano e não têm ponto em comum ($r \cap s = \varnothing$).
- **Não (retas não coplanares):**
  - **reversas**: não existe plano que contenha as duas. Elas nunca se encontram, mas também não são paralelas.

**Pergunta 2: quantos pontos elas têm em comum?** É ela que separa os três casos coplanares: todos (coincidentes), um (concorrentes) ou nenhum (paralelas).

> 📌 **Definição.** Duas retas são **reversas** quando não existe um plano que contenha ambas.

> ⚠️ **Erro comum.** Pensar que "retas que não se encontram são paralelas". No espaço, isso não basta: paralelas precisam, **além disso**, estar num mesmo plano. Retas reversas também não se encontram e não são paralelas.

🎯 **No dia a dia**
- Os dois trilhos de uma ferrovia reta: **paralelas**.
- Duas ruas que se cruzam numa esquina: **concorrentes**.
- Um viaduto passando por cima de uma avenida: as pistas são **reversas**.

O melhor laboratório para enxergar essas posições é um **cubo**. Cada aresta está numa reta. 

🕹️ Escolha duas arestas e descubra a posição relativa delas antes de clicar em mostrar a resposta.

> 💡 **Dica.** Para conferir sua resposta, o widget usa a função `u3d.classificar_posicao_retas`. Ela faz contas com vetores, um assunto do módulo 07 (Geometria Analítica). Aqui ela serve apenas como "professor" que confere o seu raciocínio.

In [12]:
vertices = u3d.vertices_do_cubo()
arestas = ["AB", "BC", "CD", "DA", "EF", "FG", "GH", "HE", "AE", "BF", "CG", "DH"]


def desenhar_arestas(aresta_1: str, aresta_2: str, mostrar_resposta: bool) -> None:
    """Desenha o cubo e as retas que contêm as duas arestas escolhidas."""
    a1, a2 = vertices[aresta_1[0]], vertices[aresta_1[1]]
    b1, b2 = vertices[aresta_2[0]], vertices[aresta_2[1]]

    fig = u3d.criar_figura(titulo=f"Retas {aresta_1} e {aresta_2}")
    fig = u3d.desenhar_cubo(fig)

    # As arestas viram retas: trechos bem maiores que o cubo, para você ver que,
    # mesmo prolongadas, retas reversas nunca se encontram.
    fig = u3d.desenhar_reta_por_dois_pontos(
        fig, a1, a2, rotulo="r", cor=u3d.CORES["reta"], comprimento=8, espessura=6
    )
    fig = u3d.desenhar_reta_por_dois_pontos(
        fig, b1, b2, rotulo="s", cor=u3d.CORES["reta_secundaria"], comprimento=8,
        espessura=6,
    )

    posicao = u3d.classificar_posicao_retas(a1, a2, b1, b2)
    if mostrar_resposta and posicao in ("concorrentes", "paralelas"):
        # Coplanares e distintas: desenhamos o plano que contém as duas. Ele passa
        # pelos dois pontos de r e por um ponto de s que esteja fora de r.
        try:
            fig = u3d.desenhar_plano_por_tres_pontos(fig, a1, a2, b1, opacidade=0.2)
        except ValueError:
            # b1 está em r (os três pontos são colineares); usamos o outro ponto de s.
            fig = u3d.desenhar_plano_por_tres_pontos(fig, a1, a2, b2, opacidade=0.2)

    u3d.mostrar(fig)

    if mostrar_resposta:
        print(f"As retas {aresta_1} e {aresta_2} são {posicao}.")
    else:
        print("Pense: elas estão num mesmo plano? Elas se encontram?")


interact(
    desenhar_arestas,
    aresta_1=widgets.Dropdown(options=arestas, value="AB", description="Aresta 1"),
    aresta_2=widgets.Dropdown(options=arestas, value="CG", description="Aresta 2"),
    mostrar_resposta=widgets.Checkbox(value=False, description="Mostrar resposta"),
);

interactive(children=(Dropdown(description='Aresta 1', options=('AB', 'BC', 'CD', 'DA', 'EF', 'FG', 'GH', 'HE'…

**Pare e pense**

1. Quantas arestas do cubo são reversas à aresta $AB$? Use o widget para conferir.
2. Duas retas reversas podem estar, cada uma, num plano horizontal? Pense no viaduto.

<details>
<summary>Ver respostas</summary>

1. Quatro: $CG$, $DH$, $EH$ e $FG$.
2. Sim. A pista de baixo está no plano do chão; a de cima, num plano horizontal mais alto. Cada uma está no seu plano, mas não existe um único plano com as duas.
</details>

> 📎 **Conexão.** Retas reversas reaparecem no notebook 02 ([Paralelismo](0602a_paralelismo.ipynb)), quando estudarmos o ângulo entre elas, e no notebook 04, quando calcularmos a distância entre duas retas reversas.

## 6. Interseção de dois planos

Olhe para o canto de uma sala: a parede encontra o teto ao longo de uma linha reta. A capa de um livro encontra o miolo ao longo da lombada, também uma reta. Isso não é coincidência.

**Postulado da interseção**
> Se dois planos distintos têm um ponto em comum, então eles têm uma reta em comum, que passa por esse ponto.

Daí concluímos as possibilidades para dois planos $\alpha$ e $\beta$:

| Situação | Interseção | Nome |
|---|---|---|
| São o mesmo plano | o próprio plano | **coincidentes** |
| Distintos, com ponto comum | uma **reta** | **secantes** (ou concorrentes) |
| Distintos, sem ponto comum | vazia ($\alpha \cap \beta = \varnothing$) | **paralelos** |


> 💡 **Dica.** Dois planos distintos **nunca** se encontram num único ponto só. Se encontram num ponto, encontram-se numa reta inteira.

🕹️ Use os controles: o primeiro sobe e desce o plano horizontal $\alpha$; o segundo inclina o plano $\beta$. Veja como a reta de interseção (em destaque) se move, e o que acontece quando $\beta$ fica horizontal.

In [13]:
def desenhar_dois_planos(altura_de_alfa: float, inclinacao_de_beta: int) -> None:
    """Desenha o plano horizontal α, o plano inclinado β e a interseção deles."""
    theta = math.radians(inclinacao_de_beta)

    # Seno e cosseno (módulo 05) servem só para girar o desenho de β: com 0° ele
    # fica horizontal, com 90° fica em pé. β sempre passa pelo ponto (0, 0, 0).
    normal_de_beta = (math.sin(theta), 0, math.cos(theta))

    # Para a reta de interseção aparecer dentro dos dois pedaços desenhados,
    # centramos os dois pedaços num ponto comum a α e β. Uma rampa com inclinação
    # theta atinge a altura de α depois de andar altura / tg(theta) na horizontal.
    if inclinacao_de_beta == 0:
        centro = (0, 0, altura_de_alfa)
    else:
        centro = (-altura_de_alfa / math.tan(theta), 0, altura_de_alfa)

    alfa = (centro, (0, 0, 1))
    beta = (centro if inclinacao_de_beta != 0 else (0, 0, 0), normal_de_beta)

    fig = u3d.criar_figura(titulo="Dois planos")
    fig = u3d.desenhar_plano(fig, *alfa, rotulo="α", cor=u3d.CORES["plano"])
    fig = u3d.desenhar_plano(
        fig, *beta, rotulo="β", cor=u3d.CORES["plano_secundario"]
    )

    if not u3d.planos_sao_paralelos(alfa, beta):
        fig = u3d.desenhar_intersecao_de_planos(fig, alfa, beta, rotulo="r")
        mensagem = "α e β são secantes: a interseção é a reta r."
    elif altura_de_alfa != 0:
        mensagem = "α e β são paralelos: não há interseção."
    else:
        mensagem = "α e β são coincidentes: são o mesmo plano."

    u3d.mostrar(fig)
    print(mensagem)


interact(
    desenhar_dois_planos,
    altura_de_alfa=widgets.FloatSlider(
        min=-2, max=2, step=0.5, value=0, description="altura de α",
        continuous_update=False,
    ),
    inclinacao_de_beta=widgets.IntSlider(
        min=0, max=90, step=10, value=60, description="inclinação de β (°)",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='altura de α', max=2.0, min=…

### E três planos?

O canto de uma sala é o encontro de **três** planos: duas paredes e o chão. Cada par se encontra numa reta, e as três retas se encontram num único ponto, o canto. Gire a figura e encontre as três retas.

In [14]:
# Cada plano é dado por (ponto, normal). Os pedaços começam no canto O e se
# estendem para dentro da sala, por isso o ponto de cada um fica deslocado.
chao = ((1.5, 1.5, 0), (0, 0, 1))
parede_1 = ((0, 1.5, 1.5), (1, 0, 0))
parede_2 = ((1.5, 0, 1.5), (0, 1, 0))

fig = u3d.criar_figura(titulo="O canto da sala: três planos e um ponto")
fig = u3d.desenhar_plano(fig, *chao, rotulo="α", tamanho=3, opacidade=0.3)
fig = u3d.desenhar_plano(
    fig, *parede_1, rotulo="β", cor=u3d.CORES["plano_secundario"], tamanho=3,
    opacidade=0.3,
)
fig = u3d.desenhar_plano(
    fig, *parede_2, rotulo="γ", cor=u3d.CORES["plano_terciario"], tamanho=3,
    opacidade=0.3,
)

# As três retas de interseção, uma para cada par de planos.
for plano_1, plano_2 in [(chao, parede_1), (chao, parede_2), (parede_1, parede_2)]:
    fig = u3d.desenhar_intersecao_de_planos(fig, plano_1, plano_2, comprimento=3)

fig = u3d.desenhar_ponto(fig, (0, 0, 0), rotulo="O", cor=u3d.CORES["destaque"])

# Câmera "dentro da sala", olhando para o canto.
fig = u3d.definir_camera(fig, olho=(1.3, 1.1, 0.8))
u3d.mostrar(fig)

**Pare e pense**

1. Dois planos distintos podem ter exatamente dois pontos em comum?
2. Três planos distintos sempre se encontram num único ponto, como no canto da sala? Pense nas páginas de um livro aberto.

<details>
<summary>Ver respostas</summary>

1. Não. Se têm um ponto em comum, têm uma reta inteira (infinitos pontos). E dois pontos já determinam essa reta.
2. Não. As páginas de um livro são vários planos que se encontram todos na mesma reta (a lombada). Três planos também podem ser paralelos, ou se cortar dois a dois em três retas paralelas (como as faces laterais de um prisma triangular).
</details>

> 📎 **Conexão.** Planos paralelos são o assunto do notebook 02 ([Paralelismo](0602a_paralelismo.ipynb)). A "abertura" entre dois planos secantes vai virar o **diedro** (notebook 05), e o canto da sala é um **triedro** (notebook 06).

## Resumo

**Conceitos primitivos:** ponto ($A$), reta ($r$), plano ($\alpha$), sem definição. Ponto **pertence** ($A \in \alpha$); reta **está contida** ($r \subset \alpha$).

**Postulados**
- Existência: há infinitos pontos numa reta, num plano e fora deles.
- Dois pontos distintos determinam uma única reta.
- Três pontos não colineares determinam um único plano.
- Inclusão: dois pontos distintos de uma reta no plano ⟹ a reta está contida no plano.
- Interseção: dois planos distintos com um ponto comum têm uma reta comum.

**Um plano fica determinado por**
1. três pontos não colineares;
2. uma reta e um ponto fora dela;
3. duas retas concorrentes;
4. duas retas paralelas distintas.

**Duas retas**

| Coplanares? | Pontos comuns | Posição |
|---|---|---|
| sim | todos | coincidentes |
| sim | um | concorrentes |
| sim | nenhum | paralelas |
| **não** | nenhum | **reversas** |

**Dois planos:** coincidentes (mesmo plano), secantes (interseção é uma reta) ou paralelos (sem interseção).

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**.

O gabarito comentado está em [`0601b_introducao_geometria_de_posicao_gabarito.ipynb`](0601b_introducao_geometria_de_posicao_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Por três pontos quaisquer passa um único plano.
# (b) Dois pontos distintos determinam uma única reta.
# (c) Duas retas que não têm ponto em comum são paralelas.
# (d) Se dois pontos distintos de uma reta pertencem a um plano, a reta está
#     contida nele.
# (e) Dois planos distintos podem ter um único ponto em comum.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": False, "b": True, "c": False, "d": True, "e": False}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo. Releia os "
        "postulados e as posições relativas de retas e de planos "
        "(seção 3 - Postulados: as regras do jogo; seções 5 e 6)."
    )
print("Certo!")

Use o cubo abaixo para o Exercício 2. Gire a figura à vontade!

In [15]:
fig = u3d.criar_figura(titulo="Cubo ABCDEFGH")
fig = u3d.desenhar_cubo(fig)
u3d.mostrar(fig)

In [ ]:
# Exercício Básico 2: classifique a posição relativa das retas que contêm cada par
# de arestas do cubo acima. Responda com "coincidentes", "concorrentes",
# "paralelas" ou "reversas".
posicoes_ex2 = {
    "AB e EF": ...,
    "AB e AD": ...,
    "AB e GH": ...,
    "AE e CD": ...,
    "BF e DH": ...,
}

print(posicoes_ex2)

In [ ]:
# Verificação
gabarito_ex2 = {
    "AB e EF": "paralelas",
    "AB e AD": "concorrentes",
    "AB e GH": "paralelas",
    "AE e CD": "reversas",
    "BF e DH": "paralelas",
}
for par, certa in gabarito_ex2.items():
    assert posicoes_ex2[par] == certa, (
        f"Tente de novo, revise: a classificação de {par} não está certa. Primeiro "
        "pergunte se existe um plano com as duas retas "
        "(seção 5 - Posições relativas de duas retas)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: quantos planos passam ao mesmo tempo por todos os elementos
# dados? Responda com um número inteiro ou com o texto "infinitos".
# (a) uma reta e um ponto fora dela
# (b) três pontos colineares
# (c) duas retas concorrentes
# (d) duas retas reversas
planos_ex3 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
}

print(planos_ex3)

In [ ]:
# Verificação
gabarito_ex3 = {"a": 1, "b": "infinitos", "c": 1, "d": 0}
for item, certa in gabarito_ex3.items():
    assert planos_ex3[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo. Releia os quatro "
        "modos de determinar um plano e a definição de retas reversas "
        "(seção 4 - Determinação de um plano; seção 5)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: considere pontos no espaço sem três colineares e sem
# quatro coplanares. Quantos planos são determinados por (a) 4 pontos? (b) 5 pontos?
# Dica: você pode reutilizar a função contar_retas_e_planos da seção 3.
planos_4_pontos_ex4 = ...
planos_5_pontos_ex4 = ...

print(planos_4_pontos_ex4, planos_5_pontos_ex4)

In [ ]:
# Verificação
assert planos_4_pontos_ex4 == 4, (
    "Tente de novo, revise: com 4 pontos, cada trio determina um plano; pense nas "
    "faces do tetraedro (seção 3 - Postulados: as regras do jogo)."
)
assert planos_5_pontos_ex4 == 10, (
    "Tente de novo, revise: conte os trios com i < j < k, ou use "
    "contar_retas_e_planos(5) (seção 3 - Postulados: as regras do jogo)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Três retas concorrentes duas a duas, em três pontos distintos, são sempre
#     coplanares.
# (b) Três retas que passam todas por um mesmo ponto são sempre coplanares.
respostas_ex5 = {
    "a": ...,
    "b": ...,
}

print(respostas_ex5)

In [ ]:
# Verificação
gabarito_ex5 = {"a": True, "b": False}
for item, certa in gabarito_ex5.items():
    assert respostas_ex5[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo. Desenhe as retas! "
        "Use o cubo: as arestas AB, AD e AE passam todas por A "
        "(seção 4 - Determinação de um plano)."
    )
print("Certo!")

In [ ]:
# Exercício Intermediário 6: qual é o MAIOR número de retas que podem ser obtidas
# como interseções de três planos distintos, tomados dois a dois?
maximo_de_retas_ex6 = ...

print(maximo_de_retas_ex6)

In [ ]:
# Verificação
assert maximo_de_retas_ex6 == 3, (
    "Tente de novo, revise: pense nas três faces laterais de uma caixa triangular "
    "(prisma), ou no canto da sala (seção 6 - Interseção de dois planos)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: quantos pares de arestas reversas existem no cubo?
# Escreva um código que percorra TODOS os pares de arestas (com laços aninhados,
# sem repetir pares) e use u3d.classificar_posicao_retas para contar quantos pares
# são "reversas". A lista de arestas e o dicionário de vértices já estão prontos.
# Lembre: para a aresta "AB", os pontos são vertices["A"] e vertices["B"].
vertices = u3d.vertices_do_cubo()
arestas = ["AB", "BC", "CD", "DA", "EF", "FG", "GH", "HE", "AE", "BF", "CG", "DH"]

pares_reversos_ex7 = 0

# complete aqui
...

print(pares_reversos_ex7)

In [ ]:
# Verificação
assert pares_reversos_ex7 == 24, (
    "Tente de novo, revise: cada aresta tem 4 arestas reversas a ela, mas cuidado "
    "para não contar o mesmo par duas vezes; use j começando em i + 1 "
    "(seção 5 - Posições relativas de duas retas)."
)
print("Certo!")

In [ ]:
# Exercício Desafio 8: um plano divide o espaço em 2 regiões. Dois planos secantes
# dividem em 4. Qual é o MAIOR número de regiões em que três planos podem dividir
# o espaço?
maximo_de_regioes_ex8 = ...

print(maximo_de_regioes_ex8)

In [ ]:
# Verificação
assert maximo_de_regioes_ex8 == 8, (
    "Tente de novo, revise: volte ao canto da sala; os três planos prolongados para "
    "todos os lados formam quantas 'salas'? (seção 6 - Interseção de dois planos)."
)
print("Certo!")

# Os três planos do canto da sala, agora prolongados para todos os lados
# (pedaços centrados no canto O).
chao = ((0, 0, 0), (0, 0, 1))
parede_1 = ((0, 0, 0), (1, 0, 0))
parede_2 = ((0, 0, 0), (0, 1, 0))

fig = u3d.criar_figura(titulo="Três planos dividindo o espaço em 8 regiões")
fig = u3d.desenhar_plano(fig, *chao, rotulo="α", tamanho=4, opacidade=0.25)
fig = u3d.desenhar_plano(
    fig, *parede_1, rotulo="β", cor=u3d.CORES["plano_secundario"], tamanho=4,
    opacidade=0.25,
)
fig = u3d.desenhar_plano(
    fig, *parede_2, rotulo="γ", cor=u3d.CORES["plano_terciario"], tamanho=4,
    opacidade=0.25,
)
for plano_1, plano_2 in [(chao, parede_1), (chao, parede_2), (parede_1, parede_2)]:
    fig = u3d.desenhar_intersecao_de_planos(fig, plano_1, plano_2, comprimento=4)
fig = u3d.desenhar_ponto(fig, (0, 0, 0), rotulo="O")
u3d.mostrar(fig)

## Conexões

**Isso depende de:**
- [`00_introducao_python.ipynb`](../00_fundamentos_e_python/00_introducao_python.ipynb) (variáveis, funções, laços `for` e f-strings)
- [`0401a_nocoes_e_proposicoes_primitivas.ipynb`](../04_geometria_plana/0401a_nocoes_e_proposicoes_primitivas.ipynb) (ponto, reta e plano como noções primitivas; postulados de determinação)
- [`0403a_angulos.ipynb`](../04_geometria_plana/0403a_angulos.ipynb) (ângulos)
- [`0406a_paralelismo.ipynb`](../04_geometria_plana/0406a_paralelismo.ipynb) (retas paralelas e concorrentes no plano)

**Isso será usado em:**
- [`0602a_paralelismo.ipynb`](0602a_paralelismo.ipynb) (retas e planos paralelos no espaço, retas reversas e ângulo entre retas)
- [`0603a_perpendicularidade.ipynb`](0603a_perpendicularidade.ipynb) (retas e planos perpendiculares)
- [`0604a_projecoes_distancias_e_angulos.ipynb`](0604a_projecoes_distancias_e_angulos.ipynb) (distância entre duas retas reversas)
- [`0605a_diedros.ipynb`](0605a_diedros.ipynb) (a "abertura" entre dois planos secantes)
- [`0606a_triedros_e_angulos_poliedricos.ipynb`](0606a_triedros_e_angulos_poliedricos.ipynb) (o canto da sala)
- 📌 Geometria Analítica Plana e Espacial (equações de retas e planos; os vetores usados em `u3d.classificar_posicao_retas`)
- 📌 Análise Combinatória (a contagem de retas e planos sem listar tudo)